<!-- [변경 이력 시작]
  2026-10-03  colab_run.ipynb로 최초 생성(S0 전용 학습 노트북)
  2026-10-07  S1 연동으로 colab_run.ipynb를 다시 만들기 전에 이전 내용을 이 이름으로 백업(내용은 그대로)
  2026-10-07  변경 이력 주석 추가
[변경 이력 끝] -->

# SELD 챌린지 베이스라인 실행 노트북

위에서부터 **셀을 순서대로** 실행하고, 2번 셀의 **설정 칸**만 바꾸면 됩니다. 자세한 설명은 `README_baseline.md`를 보세요.
런타임은 **GPU**(런타임 → 런타임 유형 변경)로 설정해 주세요. 세션이 끊기면 같은 셀을 다시 실행하면 이어서 합니다(`resume`).

## 1. 드라이브 연결, 폴더 위치, 패키지 설치
드라이브를 연결하고 코드 폴더로 이동합니다. 1분 이내.
`DRIVE_ROOT`는 `dataset/`, `internal/`, `baseline/`이 들어 있는 프로젝트 폴더 경로입니다. 다르면 고치세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/ASR_2026-2/for_dataset"   #@param {type:"string"}
CODE_DIR = f"{DRIVE_ROOT}/baseline/seld_challenge"

import os
os.chdir(CODE_DIR)
!pip install -q librosa soundfile pyyaml tqdm
!nvidia-smi -L

## 2. 설정 칸 (여기만 바꾸세요)
- `dataset`: 학습할 데이터셋 (`main20` = 카메라가 도는 20초, `ctrl5` = 카메라 고정 5초)
- `preset`: B1 = 소리만 / B2 = 영상+소리 / B2-0 = B2에서 영상을 0으로 / B3 = B2 + 화면 전체 + 시간 창
- `batch_size`: `auto`면 main20 64, ctrl5 256. GPU 메모리가 모자라면 숫자로 줄이세요.
- `eval_video`: 평가 때 영상을 `real`(그대로) / `zeros`(0) / `shuffle`(다른 클립 영상)로 바꿉니다.
- `eval_dataset`: 비우면 학습한 데이터셋으로 평가합니다. 다른 데이터셋으로 평가하려면 고르세요(예: fix20).

In [ ]:
dataset = "main20"       #@param ["main20", "ctrl5"]
preset = "B2"            #@param ["B1", "B2", "B2-0", "B3"]
seed = 1                 #@param {type:"integer"}
epochs = 200             #@param {type:"integer"}
batch_size = "auto"      #@param {type:"string"}
lr = 0.001               #@param {type:"number"}
eval_video = "real"      #@param ["real", "zeros", "shuffle"]
eval_dataset = ""        #@param ["", "main20", "ctrl5", "fix20"]
exclude_list = ""        #@param ["", "configs/exclude_val_fix20.txt"]

import shlex
_sets = {"drive_root": DRIVE_ROOT, "seed": seed, "epochs": epochs, "batch_size": batch_size, "lr": lr, "eval_video": eval_video}
if eval_dataset: _sets["eval_dataset"] = eval_dataset
if exclude_list: _sets["exclude_list"] = exclude_list
ARGS = shlex.join(["--dataset", dataset, "--preset", preset] + [x for k, v in _sets.items() for x in ("--set", f"{k}={v}")])
print(ARGS)

## 3. prepare: 데이터 풀기
드라이브의 zip을 코랩 로컬 디스크(`/content/data`)에 풉니다. 이미 풀려 있으면 건너뜁니다. 처음에는 **수 분**(main20 약 12GB) 걸릴 수 있습니다. 세션이 새로 시작되면 다시 풀어야 합니다.

In [ ]:
!python run.py prepare {ARGS}

## 4. features: 특징 뽑기
오디오와 영상 특징을 뽑아 **드라이브에 저장**(캐시)합니다. 이미 있으면 건너뛰므로, 처음 한 번만 오래 걸립니다(영상 특징은 GPU로도 **수십 분**, 미측정).
오디오 특징은 프리셋과 무관하게 한 번만, 영상 특징은 `video_crop`(center/full)마다 따로 저장됩니다.

In [ ]:
!python run.py features {ARGS}

## 5. train: 학습
학습합니다. 매 에폭 `last.pth`(이어 하기용)와 `best_model.pth`(val 점수가 가장 좋은 모델)를 드라이브에 저장하고 `train_log.csv`에 기록합니다.
**가장 오래 걸리는 셀**입니다(미측정). 세션이 끊기면 이 셀을 다시 실행하면 끊긴 에폭부터 이어 합니다.

In [ ]:
!python run.py train {ARGS}

## 6. eval, swaptest: val 점수
`eval`은 가장 좋은 모델로 val을 채점합니다. `swaptest`(영상+소리 모델만)는 같은 모델에 **진짜 영상 / 다른 클립 영상 / 0 영상**을 넣어 점수를 비교합니다(영상을 실제로 쓰는지 확인). 각각 수 분 이내.
소리만 쓰는 모델(B1)은 `swaptest`가 에러를 내는 것이 정상입니다.

In [ ]:
!python run.py eval {ARGS}
if preset != "B1":
    !python run.py swaptest {ARGS}

## 7. predict: test 예측과 제출 파일
test 입력으로 예측 csv를 만들고 제출용 zip(`submission_*.zip`)으로 묶습니다. 수 분 이내. test 정답은 없으므로 점수는 나오지 않습니다(주최자가 채점).

In [ ]:
!python run.py predict {ARGS}

## 8. summary: 결과 표 보기
지금까지 돌린 모든 실행을 `results/summary.csv` 한 표로 모으고, `results/gaps.csv`에 **"영상 덕분에 오른 점수"**(B2−B1, B2−(B2-0), B3−B1)를 계산합니다. 몇 초.

In [ ]:
!python run.py summary {ARGS}
import pandas as pd
RES = f"{DRIVE_ROOT}/seld_runs/results"
display(pd.read_csv(f"{RES}/summary.csv"))
if os.path.exists(f"{RES}/gaps.csv"):
    display(pd.read_csv(f"{RES}/gaps.csv"))